In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from PIL import Image as PILImage
from sklearn.model_selection import train_test_split

from datasets.mvtec_dataset import MVTecDataset
from datasets.realiad_dataset import RealIadDataset
from moviad.utilities.configurations import TaskType, Split, LabelName

In [2]:
dataset_path_mvtec = "/mnt/disk1/borsattifr/datasets/mvtec"
dataset_path_realiad = "/mnt/disk1/yfbenkhalifa/datasets/realiad/realiad_256"

## Manually check concept annotation

In [4]:
carpet_df = pd.read_csv("/mnt/disk1/arianna_stropeni/cbm_data/mvtec/carpet_dataset_automated.csv")
carpet_concepts = [col for col in carpet_df.columns if col not in ["image_path", "mask_path", "label_index", "anomaly_type", "split"]]

In [ ]:
#automatically exclude anomalous concepts
anomalous_concepts = ['Dark Area', 'Uneven Distribution', 'Color Variation', 'Fiber Protrusion', 'Hole', 'Metallic Sheen', 'Unraveling', 'Texture Disruption', 'Chromatic Anomaly', 'Pattern Disruption', 'Detached Fiber', 'Damaged Fiber']

carpet_df.loc[carpet_df["label_index"] == 0, anomalous_concepts] = 0

In [ ]:
def show_image(index):
    print(f"Rendering index {index}")
    row = carpet_df.iloc[index]

    img = PILImage.open(row["image_path"])
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Label: {row['label_index']}")
    plt.show()

    # Create input widgets for each editable feature
    editors = {}
    for col in carpet_concepts:
        value = row[col]
        if isinstance(value, (int, float)):
            editor = widgets.FloatText(value=value, description=col)
        else:
            editor = widgets.Text(value=str(value), description=col)
        editors[col] = editor

    # Button to save edits
    save_button = widgets.Button(description="Save Changes", button_style='success')

    def save_changes(b):
        for col, widget in editors.items():
            carpet_df.at[index, col] = widget.value
        print(f"✅ Changes saved for index {index}")

    save_button.on_click(save_changes)

    # Display form
    display(widgets.VBox(list(editors.values()) + [save_button]))


# Interface to scroll through samples
widgets.interact(show_image, index=widgets.IntSlider(min=0, max=len(carpet_df)-1, step=1))

In [ ]:
carpet_df.to_csv("/mnt/disk1/arianna_stropeni/cbm_data/mvtec/carpet_dataset.csv", index = False)

## Create complete dataset

### MVTec-AD

In [ ]:
#mvtec dataset
categories = ["bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather", "metal_nut", "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper"]
full_dataset_mvtec = pd.DataFrame()

for category in categories:
    for split in ["train", "test"]:
        partial_dataset = MVTecDataset(task = TaskType.SEGMENTATION, root = dataset_path_mvtec, category = category, split = split)
        partial_dataset.load_dataset()
        partial_dataset.samples["category"] = category

        full_dataset_mvtec = pd.concat([full_dataset_mvtec, partial_dataset.samples], ignore_index=True)

full_dataset_mvtec = full_dataset_mvtec.reset_index(drop=True)

### Real-IAD

In [ ]:
#real-iad dataset
categories = ["switch", "u_block", "usb", "woodstick", "rolled_strip_base", "toy", "eraser", "usb_adaptor", "fire_hood", "regulator", "porcelain_doll", "toothbrush", "wooden_beads", "button_battery", "mounts",
              "plastic_nut", "pcb", "end_cap", "phone_battery", "bottle_cap", "mint", "toy_brick", "plastic_plug", "zipper", "audiojack", "terminalblock", "tape", "transistor1", "sim_card_set", "vcpill"]
full_dataset_realiad = pd.DataFrame()

for category in categories:
    partial_dataset = RealIadDataset(root = dataset_path_realiad, category=category)
    print(f"Loading {category} dataset...")
    partial_dataset.load_dataset()
    partial_dataset.samples["category"] = category
    full_dataset_realiad = pd.concat([full_dataset_realiad, partial_dataset.samples], ignore_index=True)

full_dataset_realiad = full_dataset_realiad.reset_index(drop=True)

In [8]:
full_dataset_realiad["stratify_key"] = full_dataset_realiad["category"].astype(str) + "_" + full_dataset_realiad["label_index"].astype(str)

In [ ]:
train_df, val_test_df = train_test_split(full_dataset_realiad, test_size=0.1, stratify=full_dataset_realiad["stratify_key"], shuffle = True)

val_df, test_df = train_test_split(val_test_df, test_size=0.5, stratify = val_test_df["stratify_key"], shuffle = True)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

full_dataset_realiad = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)

In [18]:
full_dataset_realiad['category_index'] = full_dataset_realiad['category'].astype('category').cat.codes

In [ ]:
full_dataset_realiad.to_csv("/mnt/disk1/arianna_stropeni/cbm_data/realiad/full_dataset.csv", index = False)

## Check differences between automated dataset and manual dataset

In [47]:
hazelnut_df = pd.read_csv("/mnt/disk1/arianna_stropeni/cbm_data/mvtec/hazelnut_dataset.csv")
hazelnut_df_automated = pd.read_csv("/mnt/disk1/arianna_stropeni/cbm_data/mvtec/hazelnut_dataset_automated.csv")
hazelnut_df_clip = pd.read_csv("/mnt/disk1/arianna_stropeni/cbm_data/mvtec/hazelnut_dataset_clip.csv")

In [ ]:
final_hazelnut_concepts = [col for col in hazelnut_df.columns if col not in ["split", "anomaly_type", "image_path", "label_index", "mask_path"]]

In [ ]:
indexed_manual = hazelnut_df.set_index("image_path")
indexed_automated = hazelnut_df_automated.set_index("image_path")
indexed_clip = hazelnut_df_clip.set_index("image_path")

In [30]:
def check_differences(df_1, df_2, column_name):
    merged_df = df_1[[column_name]].rename(columns={column_name: f"manual_{column_name}"}).join(df_2[[column_name]].rename(columns={column_name: f"automated_{column_name}"}), how = "inner")
    merged_df["differences"] = merged_df[f"manual_{column_name}"] != merged_df[f"automated_{column_name}"]

    #compute accuracy 
    accuracy = (merged_df[f'manual_{column_name}'] == merged_df[f"automated_{column_name}"]).mean()

    #compute precision and recall
    TP = ((merged_df[f'manual_{column_name}'] == 1) & (merged_df[f"automated_{column_name}"] == 1)).sum()
    TN = ((merged_df[f'manual_{column_name}'] == 0) & (merged_df[f"automated_{column_name}"] == 0)).sum()
    FP = ((merged_df[f'manual_{column_name}'] == 0) & (merged_df[f"automated_{column_name}"] == 1)).sum()
    FN = ((merged_df[f'manual_{column_name}'] == 1) & (merged_df[f"automated_{column_name}"] == 0)).sum()

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0

    # print(f"Accuracy of automated concepts for concept {column_name}: {accuracy:.2f}")
    # print(f"Precision of automated concepts for concept {column_name}: {precision:.2f}")
    # print(f"Recall of automated concepts for concept {column_name}: {recall:.2f}")

    return accuracy, precision, recall

In [ ]:
def compute_avg_metrics(concepts, df_1, df_2, concept_type):
    all_acc = []
    all_prec = []
    all_rec = []

    for concept in concepts:
        accuracy, precision, recall = check_differences(df_1, df_2, concept)
        all_acc.append(accuracy)
        all_prec.append(precision)
        all_rec.append(recall)

    avg_acc, std_acc = np.mean(all_acc), np.std(all_acc)
    avg_prec, std_prec = np.mean(all_prec), np.std(all_prec)
    avg_rec, std_rec = np.mean(all_rec), np.std(all_rec)

    print(f"Average accuracy of {concept_type} concepts: {avg_acc:.2f} ± {std_acc:.2f}")
    print(f"Average precision of {concept_type} concepts: {avg_prec:.2f} ± {std_prec:.2f}")
    print(f"Average recall of {concept_type} concepts: {avg_rec:.2f} ± {std_rec:.2f}\n")

In [52]:
anomaly_concepts_obj = ["Crack", "Hole", "Split", "Fracture", "Visible Mark", "Visible Crevices", "Dark Interior", "Broken Shell", "Visible Ink"]
normal_concepts_obj = ["Intact Shell"]
anomaly_concepts_subj = ["Surface Imperfection", "Texture Anomaly", "Structural Damage", "Irregular Shape", "Surface Discontinuity", "Linear Disruption", "Uneven Tone", "Color Deviation"]
normal_concepts_subj = ["Compact Appearance", "Natural Appearance", "Uniform Appearance"]

In [ ]:
anomaly_concepts_obj = ["Hole", "Dark Area", "Metallic Sheen", "Lighter Area", "Shadowed Area", "Unraveling", "Chromatic Anomaly", "Isolated Patch"]
anomaly_concepts_subj = ["Color Variation", "Uneven Distribution", "Irregular Shape", "Spotty Appearance", "Discoloration", "Pattern Disruption"]
uniformative_concepts = ["Rectangular Pattern", "Neutral Color"]

In [ ]:
#compute differences with gemma-annotated df
compute_avg_metrics(final_hazelnut_concepts, indexed_manual, indexed_automated, "all")
compute_avg_metrics(anomaly_concepts_obj, indexed_manual, indexed_automated, "objective anomaly-related")
compute_avg_metrics(normal_concepts_obj, indexed_manual, indexed_automated, "objective normality-related")
compute_avg_metrics(anomaly_concepts_subj, indexed_manual, indexed_automated, "subjective anomaly-related")
compute_avg_metrics(normal_concepts_subj, indexed_manual, indexed_automated, "subjective normality-related")

In [ ]:
#compute differences with clip-annotated df
compute_avg_metrics(final_hazelnut_concepts, indexed_manual, indexed_clip, "all")
compute_avg_metrics(anomaly_concepts_obj, indexed_manual, indexed_clip, "objective anomaly-related")
compute_avg_metrics(anomaly_concepts_subj, indexed_manual, indexed_clip, "subjective anomaly-related")
compute_avg_metrics(normal_concepts_obj, indexed_manual, indexed_clip, "objective normality-related")
compute_avg_metrics(normal_concepts_subj, indexed_manual, indexed_clip, "subjective normality-related")